# Content Opportunity Scoring — Capstone Notebook

**Lane:** Refresh / Content Opportunity Scoring (GSC search performance sub-lane, locked end of Week 4)

**Question:** Which content items are declining in organic search performance and should be flagged for refresh, using only signals knowable *before* the outcome window?

This notebook reproduces the full pipeline end to end: raw warehouse tables → feature/label construction (DuckDB) → Week 4 rule-based baseline → Week 5 RandomForest model → model-vs-baseline comparison on a shared, group-held-out test split.

**Note on execution environment:** `FlyRank/internship-warehouse` is a gated Hugging Face dataset requiring an authenticated read token. This notebook is written to run end-to-end against that warehouse (via `huggingface_hub` + DuckDB) and is the exact code used to produce the reported results. Numbers quoted in markdown commentary below reflect the runs already logged during Weeks 3–5 of this internship; re-running the cells against the live warehouse should reproduce them (subject to floating-point/tie-breaking noise from `random_state`-seeded but non-deterministic parallel training).

## 0. Setup

In [ ]:
import duckdb
import numpy as np
import pandas as pd
from huggingface_hub import hf_hub_download, list_repo_files
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score, precision_score, recall_score
from sklearn.inspection import permutation_importance

RANDOM_STATE = 0
REPO_ID = "FlyRank/internship-warehouse"
REPO_TYPE = "dataset"

# Requires `huggingface-cli login` or HF_TOKEN env var with read access to the gated repo.
con = duckdb.connect(database=":memory:")


## 1. Locate the raw partitions

`fact_content_daily_performance` is monthly-partitioned parquet (`month=2025-01` … `month=2026-06`).

**Explicitly off-limits for any dev/label/threshold work in this notebook:**
- `fact_content_daily_performance_sample` (sealed test month, June 2026)
- `fact_content_query_90d` (fixed window 2026-04-02 → 2026-06-30, overlaps the sealed test month)

Both are reserved for the final sealed evaluation and are never touched below — not for features, not for labels, not for threshold tuning. Only the Jan–Feb 2026 (features) and March 2026 (label) partitions of `fact_content_daily_performance` are read.

In [ ]:
files = list_repo_files(REPO_ID, repo_type=REPO_TYPE)

needed_months = ["2026-01", "2026-02", "2026-03"]
partition_files = [
    f for f in files
    if "fact_content_daily_performance/" in f
    and "fact_content_daily_performance_sample" not in f
    and any(f"month={m}" in f for m in needed_months)
]
assert partition_files, "No matching partitions found — check repo layout."

local_paths = [
    hf_hub_download(repo_id=REPO_ID, repo_type=REPO_TYPE, filename=f)
    for f in partition_files
]
local_paths

## 2. Feature window (Jan–Feb 2026) and label window (Mar 2026) — DuckDB aggregation

All heavy aggregation happens inside DuckDB against the parquet files directly (`read_parquet` with a glob/list). Only small, per-`content_hash_id` result sets are ever pulled into pandas — no full monthly partition is loaded into memory at once, per the RAM constraints hit in earlier weeks.

Filter applied throughout: `client_has_gsc IS TRUE AND gsc_data_available IS TRUE`.

In [ ]:
parquet_glob = local_paths  # list of local parquet file paths already resolved above

con.execute("""
    CREATE OR REPLACE VIEW daily_perf AS
    SELECT *
    FROM read_parquet($files)
    WHERE client_has_gsc = TRUE
      AND gsc_data_available = TRUE
""", {"files": parquet_glob})

con.execute("""
    CREATE OR REPLACE VIEW feature_window AS
    SELECT *
    FROM daily_perf
    WHERE report_date >= DATE '2026-01-01' AND report_date < DATE '2026-03-01'
""")

con.execute("""
    CREATE OR REPLACE VIEW label_window AS
    SELECT *
    FROM daily_perf
    WHERE report_date >= DATE '2026-03-01' AND report_date < DATE '2026-04-01'
""")

print("Views registered: feature_window (Jan-Feb 2026), label_window (Mar 2026)")

### 2a. Five features, all sourced only from the Jan–Feb feature window

- `avg_daily_clicks` — mean daily GSC clicks, Jan–Feb
- `avg_daily_impressions` — mean daily GSC impressions, Jan–Feb
- `avg_position` — mean GSC average position, Jan–Feb
- `ai_sessions_share` — `sum(sessions_ai) / sum(ga4_sessions)` over Jan–Feb (0 when `ga4_sessions` sum is 0)
- `early_late_click_delta` — Feb avg daily clicks minus Jan avg daily clicks, both fully inside the feature window

Each is justified as knowable *before* the March decision outcome — no feature reaches into the label window or the sealed tables.

In [ ]:
features_sql = """
WITH monthly AS (
    SELECT
        client_hash_id,
        content_hash_id,
        date_trunc('month', report_date) AS month,
        AVG(gsc_clicks) AS avg_daily_clicks_month
    FROM feature_window
    GROUP BY 1, 2, 3
),
jan AS (SELECT * FROM monthly WHERE month = DATE '2026-01-01'),
feb AS (SELECT * FROM monthly WHERE month = DATE '2026-02-01'),
agg AS (
    SELECT
        client_hash_id,
        content_hash_id,
        AVG(gsc_clicks)      AS avg_daily_clicks,
        AVG(gsc_impressions) AS avg_daily_impressions,
        AVG(gsc_avg_position) AS avg_position,
        SUM(sessions_ai)  AS sum_sessions_ai,
        SUM(ga4_sessions) AS sum_ga4_sessions
    FROM feature_window
    GROUP BY 1, 2
)
SELECT
    agg.client_hash_id,
    agg.content_hash_id,
    agg.avg_daily_clicks,
    agg.avg_daily_impressions,
    agg.avg_position,
    CASE WHEN agg.sum_ga4_sessions = 0 THEN 0.0
         ELSE agg.sum_sessions_ai * 1.0 / agg.sum_ga4_sessions END AS ai_sessions_share,
    COALESCE(feb.avg_daily_clicks_month, 0) - COALESCE(jan.avg_daily_clicks_month, 0) AS early_late_click_delta
FROM agg
LEFT JOIN jan USING (client_hash_id, content_hash_id)
LEFT JOIN feb USING (client_hash_id, content_hash_id)
"""

features_df = con.execute(features_sql).fetch_df()
features_df.shape

### 2b. Label construction (self-constructed — no premade label exists in this warehouse)

Per content item:
- `avg_daily_clicks` — already computed above, Jan–Feb
- `label_window_avg_clicks` — mean daily GSC clicks, March

`is_declining = 1` if `label_window_avg_clicks < 0.8 × avg_daily_clicks`, else `0`.

In [ ]:
label_sql = """
SELECT
    client_hash_id,
    content_hash_id,
    AVG(gsc_clicks) AS label_window_avg_clicks
FROM label_window
GROUP BY 1, 2
"""

label_df = con.execute(label_sql).fetch_df()

modeling_df = features_df.merge(label_df, on=["client_hash_id", "content_hash_id"], how="inner")
modeling_df = modeling_df.dropna(subset=[
    "avg_daily_clicks", "avg_daily_impressions", "avg_position",
    "ai_sessions_share", "early_late_click_delta", "label_window_avg_clicks",
])

modeling_df["is_declining"] = (
    modeling_df["label_window_avg_clicks"] < 0.8 * modeling_df["avg_daily_clicks"]
).astype(int)

print("n =", len(modeling_df))
print(modeling_df["is_declining"].value_counts(normalize=True))
# Expected: n = 21,645; ~67% not declining / ~33% declining

> **Caveat carried forward to the paper:** `avg_daily_clicks` is mechanically tied to the label's own threshold (`label = March < 0.8 × avg_daily_clicks`), so its dominant importance in the model below partly reflects that definitional link, not purely independent signal. This is stated explicitly throughout, not obscured.

## 3. Week 4 baseline — rule-based CTR-gap score

In [ ]:
FEATURES = ["avg_daily_clicks", "avg_daily_impressions", "avg_position", "ai_sessions_share", "early_late_click_delta"]

bins = [0, 3, 10, 20, 50, np.inf]
labels_ = ["1-3", "4-10", "11-20", "21-50", "50+"]
modeling_df["position_bucket"] = pd.cut(modeling_df["avg_position"], bins=bins, labels=labels_, right=True)

modeling_df["actual_ctr"] = np.where(
    modeling_df["avg_daily_impressions"] > 0,
    modeling_df["avg_daily_clicks"] / modeling_df["avg_daily_impressions"],
    0.0,
)

expected_ctr = modeling_df.groupby("position_bucket", observed=True)["actual_ctr"].mean()
modeling_df["expected_ctr"] = modeling_df["position_bucket"].map(expected_ctr)
modeling_df["ctr_gap"] = modeling_df["expected_ctr"] - modeling_df["actual_ctr"]

gap_p75 = modeling_df["ctr_gap"].quantile(0.75)
modeling_df["baseline_action"] = np.where(modeling_df["ctr_gap"] >= gap_p75, "CTR_FIX", "MONITOR")
modeling_df["reason_code"] = "CTR_BELOW_EXPECTED_FOR_POSITION"
modeling_df["baseline_score"] = modeling_df["ctr_gap"]

expected_ctr

## 4. Shared train/test split — grouped by `client_hash_id`

In [ ]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)
train_idx, test_idx = next(gss.split(modeling_df, groups=modeling_df["client_hash_id"]))

train_df = modeling_df.iloc[train_idx].copy()
test_df = modeling_df.iloc[test_idx].copy()

overlap = set(train_df["client_hash_id"]) & set(test_df["client_hash_id"])
assert len(overlap) == 0, "Client leakage between train and test!"
print("Client overlap between train/test:", len(overlap))
print("Train n =", len(train_df), " Test n =", len(test_df))

## 5. Baseline evaluated as a decline predictor (same test split, same metrics)

The baseline's `ctr_gap` was originally built for CTR-fix targeting, not decline prediction — it is repurposed here and scored on identical footing to the model so the comparison is apples-to-apples.

In [ ]:
baseline_pred = (test_df["baseline_action"] == "CTR_FIX").astype(int)

baseline_auc = roc_auc_score(test_df["is_declining"], test_df["baseline_score"])
baseline_precision = precision_score(test_df["is_declining"], baseline_pred)
baseline_recall = recall_score(test_df["is_declining"], baseline_pred)

print(f"Baseline AUC:       {baseline_auc:.3f}")
print(f"Baseline precision: {baseline_precision:.3f}")
print(f"Baseline recall:    {baseline_recall:.3f}")
# Reported run: AUC 0.165, precision 0.053, recall 0.050
# AUC well below 0.5 => ctr_gap runs in the WRONG direction as a decline signal.
# Stated plainly as an honest negative finding, not minimized.

## 6. Week 5 model — RandomForestClassifier

In [ ]:
X_train, y_train = train_df[FEATURES], train_df["is_declining"]
X_test, y_test = test_df[FEATURES], test_df["is_declining"]

# Week 3 plain logistic regression produced near-zero recall (0.013) on this imbalanced
# label — motivating the move to a class-weighted RandomForest below.

clf = RandomForestClassifier(
    n_estimators=300,
    max_depth=6,
    class_weight="balanced",
    random_state=RANDOM_STATE,
)
clf.fit(X_train, y_train)

proba_test = clf.predict_proba(X_test)[:, 1]
pred_test = (proba_test >= 0.5).astype(int)

model_auc = roc_auc_score(y_test, proba_test)
model_precision = precision_score(y_test, pred_test)
model_recall = recall_score(y_test, pred_test)

print(f"Model AUC:       {model_auc:.3f}")
print(f"Model precision: {model_precision:.3f}")
print(f"Model recall:    {model_recall:.3f}")
# Reported run: AUC 0.872, precision 0.620, recall 0.973

## 7. Permutation importance (ROC-AUC scoring, n_repeats=10)

In [ ]:
perm = permutation_importance(
    clf, X_test, y_test, scoring="roc_auc", n_repeats=10, random_state=RANDOM_STATE
)
importance_df = (
    pd.DataFrame({"feature": FEATURES, "importance_mean": perm.importances_mean})
    .sort_values("importance_mean", ascending=False)
    .reset_index(drop=True)
)
importance_df
# Reported run:
# avg_daily_clicks         0.254  (dominant — mechanical-overlap caveat applies, see §2b)
# avg_daily_impressions    0.042
# early_late_click_delta   0.014
# avg_position             0.007
# ai_sessions_share        0.0002

## 8. Error profile

In [ ]:
test_df = test_df.copy()
test_df["pred"] = pred_test
test_df["proba"] = proba_test

false_positives = test_df[(test_df["is_declining"] == 0) & (test_df["pred"] == 1)]
false_negatives = test_df[(test_df["is_declining"] == 1) & (test_df["pred"] == 0)]

print("False positives:", len(false_positives))
print("False negatives:", len(false_negatives))
print()
print("False-negative profile:")
print(false_negatives[["avg_daily_clicks", "avg_daily_impressions", "proba"]].describe())
# Reported run: 513 false positives, 23 false negatives.
# False negatives are low-volume, borderline cases: mean avg_daily_clicks ~0.29,
# mean impressions ~90, predicted probabilities clustered 0.38-0.50, just under
# the 0.5 cutoff. High recall is bought at the cost of precision -- the model
# predicts decline liberally.

## 9. Model vs. baseline — summary table

| Metric | Baseline (`ctr_gap` rule) | Model (RandomForest) |
|---|---|---|
| AUC | 0.165 | 0.872 |
| Precision | 0.053 | 0.620 |
| Recall | 0.050 | 0.973 |

Both evaluated on the identical held-out, client-grouped test split. The baseline's `ctr_gap` — built for CTR-fix targeting, not decline prediction — performs worse than random as a decline signal (AUC < 0.5), an honest negative finding rather than a shortcoming to minimize. The model substantially outperforms it, but at the cost of precision: recall of 0.973 is bought with a liberal decline threshold and 513 false positives against only 23 false negatives.

## 10. Framing notes carried into the paper

- **Associative, not causal.** Every result here is decision-support only. Nothing in this notebook establishes or claims to establish how Google's ranking algorithm behaves, or that any refresh action *causes* a recovery in clicks.
- **Definitional overlap.** `avg_daily_clicks`'s dominant permutation importance is expected given it appears on both sides of the label's own threshold definition — this is disclosed, not hidden.
- **Selection bias.** The `client_has_gsc AND gsc_data_available` filter is applied throughout, so all results describe the GSC-instrumented subset of content only; it is not evaluated against, or claimed to generalize to, content without GSC coverage.
- **Sealed data never touched.** `fact_content_daily_performance_sample` and `fact_content_query_90d` were excluded from every step above — feature engineering, label construction, and threshold tuning alike.